## Positional Encodings Explained

Welcome back to Deconstructing the Transformer Architecture! This is our third lesson together, and we're making excellent progress building the essential components that power modern language models. In our previous lessons, we explored multi-head attention mechanisms and the feed-forward networks with Add & Norm operations that process information within each Transformer block. Now, we turn to a fundamental challenge that every sequence model must solve: how do we help our model understand the order of elements in a sequence?

Today, we'll tackle Positional Encodings, the ingenious solution that injects sequence order information into Transformer models. Unlike RNNs, which process sequences step by step and naturally encode position through their sequential processing, Transformers process all positions simultaneously through parallel attention operations. This creates both their greatest strength (parallelizability) and a critical challenge: without explicit positional information, attention mechanisms are completely permutation-invariant. We'll implement sinusoidal positional encodings from scratch, visualize their fascinating mathematical patterns, and understand why this elegant solution has become a cornerstone of modern NLP architectures. Let's go!

---

### The Permutation Invariance Problem

The self-attention mechanism we've built in previous lessons has a remarkable property that becomes problematic when dealing with sequences: it is permutation-invariant. This means that if we shuffle the order of tokens in our input sequence, the attention mechanism would produce identical outputs for each position, just in the shuffled order. While this property enables efficient parallel processing, it completely ignores the crucial sequential nature of language.

Consider the sentences "The cat sat on the mat" and "Mat the on sat cat the." The words are identical, but their meaning is entirely different due to word order. Without positional information, our attention mechanism cannot distinguish between these fundamentally different sentences. The mathematical reason lies in how attention scores are computed:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Since this computation depends only on the content-based similarity between query and key vectors, permuting the input positions doesn't change the relative relationships between tokens.

This permutation invariance extends beyond just attention: it affects the entire information flow through the Transformer. Feed-forward networks process each position independently, and layer normalization operates across the feature dimension. Without explicit positional signals, the model has no way to learn that "The cat sat" and "Sat cat the" represent different grammatical structures and meanings.

---

### Understanding Sinusoidal Positional Encodings

Sinusoidal positional encodings solve this problem through an elegant mathematical approach that encodes position using sine and cosine functions with different frequencies. The core insight is to create a unique positional signature for each position that can be added directly to token embeddings without interfering with the learned semantic representations.

The mathematical formulation uses alternating sine and cosine functions across the model dimensions:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Where $pos$ is the position in the sequence, $i$ is the dimension index, and $d_{model}$ is the model dimension. This creates a unique pattern for each position: lower dimensions oscillate rapidly with high-frequency patterns that capture fine-grained positional differences, while higher dimensions oscillate slowly with low-frequency patterns that encode broader positional relationships. The magic lies in the frequency progression: each dimension pair uses a frequency that's a power of 10000, creating a binary-like encoding system that can theoretically represent infinite sequence lengths while maintaining unique signatures for each position.

---

### Implementing Sinusoidal Positional Encodings

Let's implement the complete `PositionalEncoding` class that creates these sinusoidal patterns:

```python
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding

        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)

        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Compute the div_term for sinusoidal pattern
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           (-math.log(10000.0) / d_model))
```

This initialization creates the foundation for our positional encodings. The `position` tensor contains values `[0, 1, 2, ..., max_len-1]` reshaped to enable broadcasting. The `div_term` computation is the key mathematical insight: it creates the frequency terms that make each dimension unique. By using `torch.arange(0, d_model, 2)`, we generate frequencies for even indices, and the exponential computation creates the geometric progression of frequencies. The use of `math.log(10000.0)` transforms the original formulation into an exponential form that's computationally stable and efficient.

Now, let's complete the implementation by applying the sine and cosine functions:

```python
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)

        # Apply cosine to odd indices (1, 3, 5, ...)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)
```

The slicing operations `pe[:, 0::2]` and `pe[:, 1::2]` elegantly separate even and odd dimensions for sine and cosine application, respectively. The conditional logic handles the case where `d_model` is odd: since `div_term` has `ceil(d_model/2)` elements but odd indices have `floor(d_model/2)` positions, we use `div_term[:-1]` to match dimensions correctly. The `register_buffer` call is crucial: it ensures the positional encodings are part of the module's state but not trainable parameters, and they'll automatically move to the correct device with the model. The forward pass simply adds the appropriate slice of positional encodings to the input embeddings, enabling the model to distinguish between different positions.

---

### Creating Token Embeddings and Integration

Before we can test our positional encodings, we need a `TokenEmbedding` class that converts token indices to dense vectors:

```python
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)

    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)
```

The `TokenEmbedding` implementation includes a critical detail often overlooked: the scaling by $\sqrt{d_{model}}$. This scaling ensures that token embeddings and positional encodings have similar magnitudes when added together. Without this scaling, either the token embeddings would dominate (if they're much larger) or the positional encodings would dominate (if token embeddings are too small), both scenarios preventing effective learning of both semantic and positional information. Xavier uniform initialization provides stable starting weights that promote healthy gradient flow during training.

---

### Visualizing and Testing Our Implementation

Let's create the complete testing pipeline that generates the expected output and visualization:

```python
def visualize_positional_encoding():
    """Visualize positional encodings as heatmap"""
    print("Visualizing Positional Encodings...")

    d_model = 64
    max_len = 500

    # Create positional encoding
    pe_layer = PositionalEncoding(d_model, max_len, dropout=0.0)
    pe_matrix = pe_layer.pe.squeeze(0)  # (max_len, d_model)

    # Create visualization
    plt.figure(figsize=(12, 8))
    sns.heatmap(pe_matrix.numpy(), cmap='RdBu', center=0)
    plt.title('Sinusoidal Positional Encodings')

    print(f"Positional encoding shape: {pe_matrix.shape}")
    print(f"Value range: [{pe_matrix.min():.4f}, {pe_matrix.max():.4f}]")

def test_embeddings_integration():
    """Test token embeddings with positional encodings"""
    print("Testing Token + Positional Embeddings...")

    vocab_size, d_model, seq_len, batch_size = 1000, 64, 12, 2
    torch.manual_seed(42)
    token_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

    token_embedding = TokenEmbedding(vocab_size, d_model)
    positional_encoding = PositionalEncoding(d_model, dropout=0.0)

    token_embeds = token_embedding(token_ids)
    final_embeds = positional_encoding(token_embeds)

    print(f"Token IDs shape: {token_ids.shape}")
    print(f"Token embeddings shape: {token_embeds.shape}")
    print(f"Final embeddings shape: {final_embeds.shape}")
    print(f"Token embedding mean: {token_embeds.mean():.4f}")
    print(f"Final embedding mean: {final_embeds.mean():.4f}")
```

This testing pipeline demonstrates the complete workflow from token indices to position-aware embeddings. When we run our complete implementation, we get the following output, which confirms our positional encodings are working correctly:

```text
Visualizing Positional Encodings...
Positional encoding shape: torch.Size([500, 64])
Value range: [-1.0000, 1.0000]
Testing Token + Positional Embeddings...
Token IDs shape: torch.Size([2, 12])
Token embeddings shape: torch.Size([2, 12, 64])
Final embeddings shape: torch.Size([2, 12, 64])
Token embedding mean: -0.0141
Final embedding mean: 0.4499
```

*Heatmap of sinusoidal positional encodings (500 tokens × 64 dims).*

The heatmap reveals the elegant mathematical structure: rows represent token positions, and columns represent model dimensions. Leftmost channels oscillate rapidly with high-frequency patterns, while rightmost channels show nearly constant values, creating a spectrum from fine-grained to coarse-grained positional information. These results demonstrate several key properties: the value range `[-1.0000, 1.0000]` confirms that our sine and cosine functions are properly bounded, preventing positional encodings from overwhelming token embeddings. The shape preservation across all transformations shows that our implementation maintains dimensional consistency throughout the pipeline. Most importantly, the change in embedding statistics (from mean `-0.0141` to `0.4499`) indicates that positional information has been successfully integrated with token representations.

---

### Conclusion and Next Steps

We've successfully implemented sinusoidal positional encodings, the elegant mathematical solution that gives Transformers their understanding of sequence order! Our implementation demonstrates how sine and cosine functions with carefully chosen frequencies create unique positional signatures that can extend to sequences longer than those seen during training. The visualization reveals the beautiful mathematical structure, where high-frequency components capture fine-grained positional differences while low-frequency components encode broader positional relationships.

With positional encodings complete, we now have all the core components needed to build a complete Transformer architecture: multi-head attention for relationship modeling, feed-forward networks for non-linear processing, Add & Norm for training stability, and positional encodings for sequence order awareness. In our next lesson, we'll begin assembling these components into complete encoder and decoder layers, bringing us closer to understanding how these individual pieces work together to create the powerful language models that have transformed NLP. Time to practice with these positional encodings!

## Building Mathematical Position Awareness

Now that you understand why positional encodings are essential for Transformers, it's time to implement the core mathematical formulas that make sinusoidal encodings work.

---

### Your Task

Your job is to complete the `PositionalEncoding` class by filling in the missing calculations. The provided code already sets up the `pe` matrix, `position` tensor, and `div_term` — you need to apply the sine and cosine functions to populate the encoding values.

Specifically, you'll need to:

- Apply sine to the even-indexed columns (0, 2, 4, ...).
- Apply cosine to the odd-indexed columns (1, 3, 5, ...).
- Handle the special case for odd `d_model` dimensions.

Remember that the sine values go to even indices and the cosine values go to odd indices, using the `position` and `div_term` tensors you've been given. Once you get this right, you'll have created the foundation that gives Transformers their sense of word order!

---

### Starter Code

```python
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding
        
        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal pattern.
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        # TODO: Apply sine to even indices of the pe matrix
        
        # TODO: Apply cosine to odd indices of the pe matrix
        # Note: The logic to handle odd d_model is provided.
        if d_model % 2 == 1:
            # TODO: Apply cosine using div_term[:-1] for odd d_model
            pass
        else:
            # TODO: Apply cosine using div_term for even d_model
            pass
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)
```

---

### Solution

Only the three TODOs in `__init__` were missing. The sine and cosine assignments become:

```python
# Apply sine to even indices (0, 2, 4, ...)
pe[:, 0::2] = torch.sin(position * div_term)

# Apply cosine to odd indices (1, 3, 5, ...)
if d_model % 2 == 1:
    pe[:, 1::2] = torch.cos(position * div_term[:-1])
else:
    pe[:, 1::2] = torch.cos(position * div_term)
```

- `position * div_term` multiplies a `(max_len, 1)` tensor by a `(ceil(d_model / 2),)` tensor. Broadcasting gives a `(max_len, ceil(d_model / 2))` matrix of angles, one for every position and every sine/cosine pair. This is the argument $pos / 10000^{2i/d_{model}}$ from the lesson's formulas.
- `pe[:, 0::2]` selects the even columns (0, 2, 4, ...), which receive `torch.sin(...)`. `pe[:, 1::2]` selects the odd columns (1, 3, 5, ...), which receive `torch.cos(...)`. Each sine/cosine pair therefore shares the same frequency.
- When `d_model` is even, there are `d_model / 2` even columns and `d_model / 2` odd columns, so `div_term` fits both assignments.
- When `d_model` is odd, there is one more even column than odd columns: `ceil(d_model / 2)` versus `floor(d_model / 2)`. `div_term` has `ceil(d_model / 2)` entries, so the cosine branch uses `div_term[:-1]` to drop the extra one and match the number of odd columns. The sine assignment always uses the full `div_term`.
- Without the fix, the `pass` statements leave `pe` all zeros. The code still runs, but every position gets the same encoding (zeros), so the model gets no position information.

This exercise has no `main()`, so the cell below just defines the full `PositionalEncoding` and `TokenEmbedding` classes with the fix applied. For example, `PositionalEncoding(64, max_len=500, dropout=0.0).pe` has shape `(1, 500, 64)`, with values between -1 and 1.

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding
        
        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal pattern.
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices (1, 3, 5, ...)
        if d_model % 2 == 1:
            # Odd d_model has one fewer odd column, so drop the last frequency
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)

## Scaling and Combining Embeddings

Now that you've implemented the sinusoidal formulas, it's time to see how token embeddings and positional encodings work together in practice.

---

### Your Task

In this exercise, you'll complete two critical missing pieces in the provided code. The `TokenEmbedding` class needs to properly scale its output, and the `PositionalEncoding` class needs to actually add positional information to the input embeddings.

You'll need to fix the `TokenEmbedding.forward` method to apply the correct scaling factor and complete the `PositionalEncoding.forward` method to combine embeddings with positional information. Take a close look at both `forward` methods — they are currently incomplete and need your attention.

Once you get both working correctly, you'll have the complete foundation for how Transformers process input sequences!

---

### Starter Code

```python
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding
        
        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal pattern.
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices (1, 3, 5, ...)
        # If d_model is odd, the slice pe[:, 1::2] has (d_model-1)/2 columns.
        # div_term has (d_model+1)/2 elements. So div_term[:-1] is used, which has (d_model-1)/2 elements.
        # If d_model is even, pe[:, 1::2] has d_model/2 columns.
        # div_term has d_model/2 elements. So div_term is used.
        # This ensures correct dimensions for both even and odd d_model.
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        # TODO: Add the positional encodings to the input embeddings
        # Hint: You need to slice self.pe to match the sequence length of x
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        # TODO: Scale the embeddings by the square root of the model dimension
        return self.embedding(x)
```

---

### Solution

Two `forward` methods had a missing step. In `PositionalEncoding.forward`, the positional encodings must be added:

```python
seq_len = x.size(1)
x = x + self.pe[:, :seq_len]
return self.dropout(x)
```

In `TokenEmbedding.forward`, the embeddings must be scaled:

```python
return self.embedding(x) * math.sqrt(self.d_model)
```

- `x.size(1)` is the sequence length of the input, which has shape `(batch_size, seq_len, d_model)`. `self.pe` has shape `(1, max_len, d_model)`, so `self.pe[:, :seq_len]` keeps only the first `seq_len` positions, giving `(1, seq_len, d_model)`.
- `x + self.pe[:, :seq_len]` broadcasts the leading dimension of 1 across the batch, so every sequence in the batch gets the same positional signal at the same position. The result keeps the shape `(batch_size, seq_len, d_model)`.
- The `PositionalEncoding.forward` stub already ended with `return self.dropout(x)`. The fix just computes the sum into `x` before that line, so dropout is applied to the combined embeddings.
- `math.sqrt(self.d_model)` is the scaling factor described in the lesson. `nn.Embedding` weights are initialized with Xavier uniform, which gives small values, so multiplying by $\sqrt{d_{model}}$ brings the token embeddings to a magnitude comparable to the positional encodings, whose values lie in $[-1, 1]$.
- Without the first fix, `PositionalEncoding.forward` returns its input unchanged (apart from dropout), so no position information reaches the model. Without the second fix, the token embeddings stay small and the positional encodings dominate the sum.

This exercise has no `main()`, so the cell below just defines the full `PositionalEncoding` and `TokenEmbedding` classes with both fixes applied. For `token_ids` of shape `(batch_size, seq_len)`, `PositionalEncoding(d_model)(TokenEmbedding(vocab_size, d_model)(token_ids))` has shape `(batch_size, seq_len, d_model)`.

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding
        
        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal pattern.
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices (1, 3, 5, ...)
        # If d_model is odd, the slice pe[:, 1::2] has (d_model-1)/2 columns.
        # div_term has (d_model+1)/2 elements. So div_term[:-1] is used, which has (d_model-1)/2 elements.
        # If d_model is even, pe[:, 1::2] has d_model/2 columns.
        # div_term has d_model/2 elements. So div_term is used.
        # This ensures correct dimensions for both even and odd d_model.
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        # Slice pe to the input's sequence length and add it to the embeddings
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        # Scale the embeddings by sqrt(d_model)
        return self.embedding(x) * math.sqrt(self.d_model)

## Debugging Faulty Encoding Logic

Now that you've implemented token and positional embeddings, it's time to test your debugging skills. The provided code contains a complete `PositionalEncoding` class, but there's a subtle bug that will create an incorrect heatmap visualization.

---

### Your Task

Run the code and examine the output. You'll notice that the generated heatmap doesn't match the expected sinusoidal pattern from the lesson. Your job is to find and fix this mathematical error.

Compare the output with the reference heatmap, then carefully review the formulas in the `PositionalEncoding` class to identify the problem. This debugging challenge will sharpen your understanding of the precise mathematical requirements for positional encodings.

---

### Starter Code

```python
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding
        
        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal pattern.
        # div_term will have shape (ceil(d_model / 2),)
        div_term = 10000.0 ** (torch.arange(0, d_model, 2).float() / d_model)
        
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices (1, 3, 5, ...)
        # If d_model is odd, the slice pe[:, 1::2] has (d_model-1)/2 columns.
        # div_term has (d_model+1)/2 elements. So div_term[:-1] is used, which has (d_model-1)/2 elements.
        # If d_model is even, pe[:, 1::2] has d_model/2 columns.
        # div_term has d_model/2 elements. So div_term is used.
        # This ensures correct dimensions for both even and odd d_model.
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)
```

---

### Solution

The bug is in the `div_term` line. The starter code computes `10000.0 ** (2i / d_model)`, but the formula divides by that value, so it must be inverted:

```python
# Buggy: the angle is multiplied by 10000^(2i/d_model)
div_term = 10000.0 ** (torch.arange(0, d_model, 2).float() / d_model)

# Fixed: the angle is divided by 10000^(2i/d_model)
div_term = 1.0 / (10000.0 ** (torch.arange(0, d_model, 2).float() / d_model))
```

- The lesson's formulas are $PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$ and $PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$. The code computes the angle as `position * div_term`, so `div_term` has to hold $\frac{1}{10000^{2i/d_{model}}}$.
- The buggy line stores $10000^{2i/d_{model}}$ itself. The angle then becomes $pos \cdot 10000^{2i/d_{model}}$, and the frequency grows with the dimension index instead of shrinking. The pattern no longer goes from fast oscillation in the first columns to slow oscillation in the last columns.
- The fix is equivalent to the lesson's version, `torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))`, because $e^{-\ln(10000) \cdot 2i / d_{model}} = 1 / 10000^{2i/d_{model}}$. Either form works.
- The bug is easy to miss because sine and cosine always return values in $[-1, 1]$. The shapes and the value range stay correct, and only the heatmap pattern reveals the problem.
- Nothing else changes: the sine and cosine assignments, the odd `d_model` handling and both `forward` methods are already correct.

This exercise has no `main()` and no plotting code, so the cell below just defines the full `PositionalEncoding` and `TokenEmbedding` classes with the fix applied. To see the heatmap, build `PositionalEncoding(64, 500, dropout=0.0).pe.squeeze(0)` and plot it, as in the lesson.

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding
        
        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal pattern.
        # div_term will have shape (ceil(d_model / 2),)
        # Fix: divide by 10000^(2i/d_model), i.e. use its reciprocal
        div_term = 1.0 / (10000.0 ** (torch.arange(0, d_model, 2).float() / d_model))
        
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices (1, 3, 5, ...)
        # If d_model is odd, the slice pe[:, 1::2] has (d_model-1)/2 columns.
        # div_term has (d_model+1)/2 elements. So div_term[:-1] is used, which has (d_model-1)/2 elements.
        # If d_model is even, pe[:, 1::2] has d_model/2 columns.
        # div_term has d_model/2 elements. So div_term is used.
        # This ensures correct dimensions for both even and odd d_model.
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)

## Runtime Error Detective Work

After debugging the frequency calculation in the previous exercise, you'll now tackle a trickier issue that arises when working with odd model dimensions. The provided code contains a bug that causes a runtime error.

---

### Your Task

When you run the code, you'll encounter a `RuntimeError` related to a tensor size mismatch. Your job is to debug this error by examining the tensor shapes involved in the positional encoding calculation. Pay close attention to how the sine and cosine functions are applied to different parts of the encoding matrix.

This exercise will sharpen your debugging skills and deepen your understanding of how tensor operations work in edge cases!

---

### Starter Code

```python
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding
        
        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal pattern.
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices (1, 3, 5, ...)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)
```

---

### Solution

The bug is in the cosine line. It always uses the full `div_term`, which only works when `d_model` is even. For an odd `d_model` it has to drop the last frequency:

```python
# Buggy: breaks when d_model is odd
pe[:, 1::2] = torch.cos(position * div_term)

# Fixed: handle both odd and even d_model
if d_model % 2 == 1:
    pe[:, 1::2] = torch.cos(position * div_term[:-1])
else:
    pe[:, 1::2] = torch.cos(position * div_term)
```

- `div_term` is built from `torch.arange(0, d_model, 2)`, so it has `ceil(d_model / 2)` entries. For `d_model = 5` that is 3 entries, and for `d_model = 64` it is 32.
- `pe[:, 0::2]` (the sine columns) has `ceil(d_model / 2)` columns, so the sine line always matches `div_term`.
- `pe[:, 1::2]` (the cosine columns) has only `floor(d_model / 2)` columns. For an even `d_model` the two counts are equal. For an odd `d_model`, the odd slice has one column fewer than `div_term` has entries.
- With `d_model = 5`, `position * div_term` has shape `(max_len, 3)`, while `pe[:, 1::2]` has shape `(max_len, 2)`. Assigning one to the other raises the `RuntimeError` about mismatched tensor sizes. The sine line is fine because `pe[:, 0::2]` has 3 columns.
- `div_term[:-1]` drops the last entry, giving `floor(d_model / 2)` entries, so the shapes match again.
- With an even `d_model`, such as the usual 64, the buggy code runs without error. That is why the bug only shows up for odd model dimensions.

This exercise has no `main()`, so the cell below just defines the full `PositionalEncoding` and `TokenEmbedding` classes with the fix applied. For example, `PositionalEncoding(5, max_len=10, dropout=0.0).pe` has shape `(1, 10, 5)` and no longer raises an error.

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        """
        Sinusoidal Positional Encoding
        
        Args:
            d_model: Model dimension
            max_len: Maximum sequence length
            dropout: Dropout rate
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal pattern.
        # div_term will have shape (ceil(d_model / 2),)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices (1, 3, 5, ...)
        # Fix: for odd d_model, pe[:, 1::2] has one column fewer than div_term has entries
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """Token Embedding layer"""
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """Convert token indices to embeddings scaled by sqrt(d_model)"""
        return self.embedding(x) * math.sqrt(self.d_model)